# 03 — Baselines & Ablation 2×2

Treina e compara os baselines da Fase 2: `DummyClassifier` (chão de
sanidade) e `LogisticRegression` em quatro variantes de feature set
(presença/ausência de `Phone Service` e `Multiple Lines` —
[ADR-005](../docs/architecture.md#adr-005--manter-features-de-sinal-fraco-para-ablation-pós-baseline)).

**Toda a lógica vive em `src/churn/training/tracking.py`** — este
notebook só orquestra os 5 runs, lê de volta via `mlflow.search_runs`
e materializa a decisão de manter ou descartar as duas features.

## 1. Setup

In [1]:
import logging
import warnings

import mlflow
import pandas as pd

from churn.config import (
    DATASET_VERSION,
    MLFLOW_EXPERIMENT_NAME,
    ROC_AUC_TARGET,
    ROOT_DIR,
    SEED,
)
from churn.data.loader import load_raw_data
from churn.data.preprocessing import (
    clean_raw,
    split_features_target,
    stratified_split,
)
from churn.models.baseline import build_dummy_baseline, build_logreg_baseline
from churn.training.tracking import log_baseline_cv_run, setup_mlflow

logging.basicConfig(level=logging.INFO, format='%(message)s')
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

# Tracking URI absoluto: ``mlruns/`` sempre na raiz do projeto,
# independentemente de onde o notebook é executado.
setup_mlflow(tracking_uri=(ROOT_DIR / 'mlruns').as_uri())

print(f'experiment   = {MLFLOW_EXPERIMENT_NAME}')
print(f'tracking uri = {mlflow.get_tracking_uri()}')
print(f'dataset      = {DATASET_VERSION}')
print(f'seed         = {SEED}')

2026/04/27 18:20:48 INFO mlflow.tracking.fluent: Experiment with name 'churn-prediction' does not exist. Creating a new experiment.


MLflow ready - experiment='churn-prediction' tracking_uri=file:///D:/Projs/Churn-Prediction-ANN/mlruns


experiment   = churn-prediction
tracking uri = file:///D:/Projs/Churn-Prediction-ANN/mlruns
dataset      = v1
seed         = 42


## 2. Carregamento dos splits

Mesmo caminho usado pelo `conftest.py`
([ADR-007](../docs/architecture.md#adr-007--fixtures-de-teste-construídas-a-partir-do-raw)):
`load_raw → clean_raw → split_features_target → stratified_split`.
Resultado: `SplitData(X_train, X_val, X_test, y_train, y_val, y_test)`
com proporções 70 / 15 / 15 estratificadas em `Churn Value`.

In [2]:
df_raw = load_raw_data()
df_clean = clean_raw(df_raw)
X, y = split_features_target(df_clean)
splits = stratified_split(X, y)

shape_table = pd.DataFrame(
    {
        'rows': [len(splits.X_train), len(splits.X_val), len(splits.X_test)],
        'churn_rate': [
            splits.y_train.mean(),
            splits.y_val.mean(),
            splits.y_test.mean(),
        ],
    },
    index=['train', 'val', 'test'],
)
shape_table

Loading raw data from D:\Projs\Churn-Prediction-ANN\data\raw\raw_data.xlsx


Loaded raw data: 7043 rows x 33 columns


Raw data schema validated successfully


Dropped 12 columns (leakage + identifiers + geo)


Coerced 'Total Charges' to float; imputed 11 blank rows with 0.0


Collapsed 'No internet service' on 6 cols and 'No phone service' on 1 col


Stratified split: train=4929 val=1057 test=1057 (target rates: 0.2654 / 0.2658 / 0.2649)


,rows,churn_rate
train,4929,0.2654
val,1057,0.2658
test,1057,0.2649


## 3. Design experimental

Cinco runs no MLflow:

| # | Run name | Phone Service | Multiple Lines | Papel |
|---|---|---|---|---|
| 1 | `dummy_baseline` | — | — | Chão de sanidade (`most_frequent`) |
| 2 | `logreg_baseline` | ✅ | ✅ | Full features — referência |
| 3 | `logreg_no_multilines_ablation` | ✅ | ❌ | Drop só `Multiple Lines` |
| 4 | `logreg_no_phone_ablation` | ❌ | ✅ | Drop só `Phone Service` |
| 5 | `logreg_no_phone_no_multilines_ablation` | ❌ | ❌ | Drop ambas |

[ADR-004](../docs/architecture.md#adr-004--colapso-de-no-internetphone-service--no)
colapsa `"No phone service"` em `"No"`, então `Multiple Lines = Yes` ⇒
`Phone Service = Yes`. Drop unilateral deixa sinal residual; só a célula
`(Phone=no, Multilines=no)` isola a contribuição **conjunta** das duas
features — daí a matriz 2×2 completa em vez de uma única ablation.

**Critério de decisão.** Manter ambas as features a menos que a remoção
conjunta produza ganho consistente em **todas** as métricas (CV + holdout,
ROC-AUC + PR-AUC) **e** o spread entre os 4 LogRegs supere o ruído de
fold (`spread(roc_auc_mean) > mean(roc_auc_std)`). Empate técnico não
é evidência de descarte seguro: o
[ADR-005](../docs/architecture.md#adr-005--manter-features-de-sinal-fraco-para-ablation-pós-baseline)
default ("nada é descartado sem evidência") prevalece.

## 4. Run 1 — `dummy_baseline`

`DummyClassifier(strategy='most_frequent')`. Não enxerga features —
serve como chão de sanidade para qualquer modelo real.

In [3]:
log_baseline_cv_run(
    model_name='dummy_baseline',
    build_pipeline=build_dummy_baseline,
    X_train=splits.X_train,
    y_train=splits.y_train,
    X_val=splits.X_val,
    y_val=splits.y_val,
    extra_tags={'phone_service': 'n/a', 'multiple_lines': 'n/a'},
)

2026/04/27 18:20:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/27 18:20:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run 'dummy_baseline' - roc_auc_mean=0.5000 holdout_val_roc_auc=0.5000


'089e284052094ae2bf3a9a9a5eb73198'

## 5. Run 2 — `logreg_baseline` (full features)

`LogisticRegression(class_weight='balanced')` sobre as 27 features
pós-one-hot. Esta run é a **referência** contra a qual as três ablations
são medidas.

In [4]:
log_baseline_cv_run(
    model_name='logreg_baseline',
    build_pipeline=lambda: build_logreg_baseline(),
    X_train=splits.X_train,
    y_train=splits.y_train,
    X_val=splits.X_val,
    y_val=splits.y_val,
    params={'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs'},
    extra_tags={'phone_service': 'yes', 'multiple_lines': 'yes'},
)

2026/04/27 18:20:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/27 18:20:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run 'logreg_baseline' - roc_auc_mean=0.8588 holdout_val_roc_auc=0.8445


'a856102819ff4df892dc8d55dcef9d97'

## 6. Ablation 2×2 (3 sub-runs)

Cada célula da matriz remove explicitamente uma combinação. O kwarg
`exclude_columns` foi adicionado a `build_preprocessing_pipeline` /
`build_logreg_baseline` para suportar a ablation sem duplicar o
`ColumnTransformer` no notebook.

### 6.1 `logreg_no_multilines_ablation` — drop só `Multiple Lines`

In [5]:
log_baseline_cv_run(
    model_name='logreg_no_multilines_ablation',
    build_pipeline=lambda: build_logreg_baseline(
        exclude_columns=('Multiple Lines',),
    ),
    X_train=splits.X_train,
    y_train=splits.y_train,
    X_val=splits.X_val,
    y_val=splits.y_val,
    params={'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs'},
    extra_tags={'phone_service': 'yes', 'multiple_lines': 'no'},
)

2026/04/27 18:21:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/27 18:21:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run 'logreg_no_multilines_ablation' - roc_auc_mean=0.8589 holdout_val_roc_auc=0.8435


'3bef401f6abd4441ac5807a7a0ca5917'

### 6.2 `logreg_no_phone_ablation` — drop só `Phone Service`

In [6]:
log_baseline_cv_run(
    model_name='logreg_no_phone_ablation',
    build_pipeline=lambda: build_logreg_baseline(
        exclude_columns=('Phone Service',),
    ),
    X_train=splits.X_train,
    y_train=splits.y_train,
    X_val=splits.X_val,
    y_val=splits.y_val,
    params={'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs'},
    extra_tags={'phone_service': 'no', 'multiple_lines': 'yes'},
)

2026/04/27 18:21:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/27 18:21:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run 'logreg_no_phone_ablation' - roc_auc_mean=0.8590 holdout_val_roc_auc=0.8448


'dd8f88c5261940b0ae2bcf60bbad1a97'

### 6.3 `logreg_no_phone_no_multilines_ablation` — drop ambas (célula crítica)

In [7]:
log_baseline_cv_run(
    model_name='logreg_no_phone_no_multilines_ablation',
    build_pipeline=lambda: build_logreg_baseline(
        exclude_columns=('Phone Service', 'Multiple Lines'),
    ),
    X_train=splits.X_train,
    y_train=splits.y_train,
    X_val=splits.X_val,
    y_val=splits.y_val,
    params={'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs'},
    extra_tags={'phone_service': 'no', 'multiple_lines': 'no'},
)

2026/04/27 18:21:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/04/27 18:21:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run 'logreg_no_phone_no_multilines_ablation' - roc_auc_mean=0.8583 holdout_val_roc_auc=0.8430


'9eeaa6f155c947978f09960b82c13185'

## 7. Comparação dos 5 runs

`mlflow.search_runs` filtra os runs pelo nome esperado e remove
duplicatas — re-execuções deste notebook geram runs novos com os mesmos
nomes, e queremos sempre comparar os mais recentes.

In [8]:
EXPECTED_RUN_NAMES = [
    'dummy_baseline',
    'logreg_baseline',
    'logreg_no_multilines_ablation',
    'logreg_no_phone_ablation',
    'logreg_no_phone_no_multilines_ablation',
]

runs_df = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    order_by=['start_time DESC'],
)
runs_df = runs_df[runs_df['tags.mlflow.runName'].isin(EXPECTED_RUN_NAMES)]
runs_df = runs_df.drop_duplicates(subset='tags.mlflow.runName', keep='first')

display_cols = [
    'tags.mlflow.runName',
    'metrics.roc_auc_mean',
    'metrics.roc_auc_std',
    'metrics.pr_auc_mean',
    'metrics.pr_auc_std',
    'metrics.f1_mean',
    'metrics.holdout_val_roc_auc',
    'metrics.holdout_val_pr_auc',
]
summary = (
    runs_df[display_cols]
    .rename(columns=lambda c: c.replace('metrics.', '').replace('tags.mlflow.', ''))
    .set_index('runName')
    .reindex(EXPECTED_RUN_NAMES)
)
summary

,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,f1_mean,holdout_val_roc_auc,holdout_val_pr_auc
runName,,,,,,,
dummy_baseline,0.5000,0.0000,0.2654,0.0004,0.0000,0.5000,0.2658
logreg_baseline,0.8588,0.0049,0.6854,0.0209,0.6434,0.8445,0.6437
logreg_no_multilines_ablation,0.8589,0.0047,0.6851,0.0204,0.6428,0.8435,0.6357
logreg_no_phone_ablation,0.8590,0.0050,0.6852,0.0218,0.6444,0.8448,0.6425
logreg_no_phone_no_multilines_ablation,0.8583,0.0049,0.6812,0.0216,0.6399,0.8430,0.6292


### 7.1 Delta vs. `logreg_baseline`

Quanto cada ablation perde (negativo) ou ganha (positivo) em relação à
referência. Critério: delta ≥ `−0.005` em `roc_auc_mean` é o gatilho
para considerar a remoção.

In [9]:
ABLATION_RUNS = [
    'logreg_no_multilines_ablation',
    'logreg_no_phone_ablation',
    'logreg_no_phone_no_multilines_ablation',
]
baseline_metrics = summary.loc['logreg_baseline']

deltas = pd.DataFrame(
    {
        'roc_auc_mean': (
            summary.loc[ABLATION_RUNS, 'roc_auc_mean']
            - baseline_metrics['roc_auc_mean']
        ),
        'pr_auc_mean': (
            summary.loc[ABLATION_RUNS, 'pr_auc_mean']
            - baseline_metrics['pr_auc_mean']
        ),
        'holdout_val_roc_auc': (
            summary.loc[ABLATION_RUNS, 'holdout_val_roc_auc']
            - baseline_metrics['holdout_val_roc_auc']
        ),
    }
)
deltas

,roc_auc_mean,pr_auc_mean,holdout_val_roc_auc
runName,,,
logreg_no_multilines_ablation,0.0001,-0.0002,-0.0010
logreg_no_phone_ablation,0.0002,-0.0002,0.0003
logreg_no_phone_no_multilines_ablation,-0.0005,-0.0042,-0.0015


## 8. Decisão final

Dois testes objetivos sobre os 5 runs:

1. **Spread vs. ruído.** O range de `roc_auc_mean` entre as 4 variantes
   de LogReg precisa exceder o desvio padrão médio dentro de cada run.
   Se `spread < mean_std`, as variantes são indistinguíveis.
2. **Coerência multi-métrica.** A célula de drop conjunto precisa
   ganhar (delta `> 0`) em **todas** as 4 métricas comparadas
   (`roc_auc_mean`, `pr_auc_mean`, `holdout_val_roc_auc`,
   `holdout_val_pr_auc`). Se mesmo uma piora, o "ganho" depende da
   métrica que se escolhe olhar — não é evidência.

Só o **AND** dos dois autoriza a remoção. Caso contrário, mantemos
([ADR-005](../docs/architecture.md#adr-005--manter-features-de-sinal-fraco-para-ablation-pós-baseline)
default).

In [10]:
LOGREG_RUNS = [
    'logreg_baseline',
    'logreg_no_multilines_ablation',
    'logreg_no_phone_ablation',
    'logreg_no_phone_no_multilines_ablation',
]

# Test 1: spread between LogReg variants vs. within-run CV noise.
roc_auc_values = summary.loc[LOGREG_RUNS, 'roc_auc_mean']
spread = float(roc_auc_values.max() - roc_auc_values.min())
mean_std = float(summary.loc[LOGREG_RUNS, 'roc_auc_std'].mean())
spread_exceeds_noise = spread > mean_std

# Test 2: joint-drop deltas across all 4 comparison metrics.
joint = 'logreg_no_phone_no_multilines_ablation'
metrics = ['roc_auc_mean', 'pr_auc_mean', 'holdout_val_roc_auc', 'holdout_val_pr_auc']
joint_drop_deltas = {
    m: float(summary.loc[joint, m] - summary.loc['logreg_baseline', m])
    for m in metrics
}
all_deltas_positive = all(d > 0 for d in joint_drop_deltas.values())

drop_authorised = spread_exceeds_noise and all_deltas_positive
verdict = 'CONSIDER drop' if drop_authorised else 'KEEP both features'

print('--- Test 1: spread vs. CV noise ---')
print(f'  spread (max - min of LogReg roc_auc_mean) = {spread:+.4f}')
print(f'  mean within-run CV std                    = {mean_std:+.4f}')
print(f'  spread > std?                             = {spread_exceeds_noise}')
print()
print('--- Test 2: joint-drop deltas vs. baseline ---')
for k, v in joint_drop_deltas.items():
    sign = 'OK' if v > 0 else 'NO'
    print(f'  {k:<24} = {v:+.4f}  [{sign}]')
print(f'  all deltas > 0?            = {all_deltas_positive}')
print()
baseline_roc_auc = summary.loc['logreg_baseline', 'roc_auc_mean']
print('--- Verdict ---')
print(f'  ROC_AUC_TARGET (project SLO) = {ROC_AUC_TARGET:.4f}')
print(f'  baseline roc_auc_mean        = {baseline_roc_auc:.4f}')
print(f'  decision                     = {verdict}')

--- Test 1: spread vs. CV noise ---
  spread (max - min of LogReg roc_auc_mean) = +0.0006
  mean within-run CV std                    = +0.0049
  spread > std?                             = False

--- Test 2: joint-drop deltas vs. baseline ---
  roc_auc_mean             = -0.0005  [NO]
  pr_auc_mean              = -0.0042  [NO]
  holdout_val_roc_auc      = -0.0015  [NO]
  holdout_val_pr_auc       = -0.0145  [NO]
  all deltas > 0?            = False

--- Verdict ---
  ROC_AUC_TARGET (project SLO) = 0.8000
  baseline roc_auc_mean        = 0.8588
  decision                     = KEEP both features


### 8.1 Conclusão

- Os 4 LogRegs são estatisticamente indistinguíveis dentro do ruído
  de CV (spread `~0.0007` ≪ std `~0.0049`).
- O "melhor" varia conforme a métrica que se escolhe olhar (ROC-AUC vs.
  PR-AUC), o que é a definição de empate técnico.
- O drop conjunto perde em todas as métricas com sinal — pequeno, mas
  consistentemente negativo.

`Phone Service` e `Multiple Lines` ficam no pipeline. Sem novo ADR:
ADR-005 já cobre essa decisão como default ("manter sob empate"). A
referência para a Fase 3 (MLP) continua sendo `logreg_baseline` com as
27 features pós-one-hot.

## 9. Feature Engineering e Bateria Sistemática

A ablação da seção 8 confirmou que `Phone Service` e `Multiple Lines` têm
sinal negligível no LogReg (spread ~0.0007, dentro do ruído de CV ±0.0049).
Com isso, temos a **referência da Fase 2**: LogReg com 27 features originais,
AUC-ROC CV = 0.8588 ± 0.0049.

A pergunta seguinte é: **features derivadas conseguem superar este teto empírico?**

Com base nos insights da EDA (`01_eda.ipynb`), cinco features de negócio foram
projetadas e implementadas em `src/churn/data/preprocessing.py`:

| Feature | Cálculo | Hipótese de negócio |
|---------|---------|---------------------|
| `risco_contrato` | Ordinal 0–2: Two year=0, One year=1, Month-to-month=2 | Contratos mensais apresentam risco de churn ~2× maior (EDA seção 4) |
| `service_count` | Qtd de add-ons ativos (0–7) | Mais serviços → maior custo de troca → menor propensão ao churn |
| `is_new` | Flag: Tenure Months ≤ 3 | Primeiros 3 meses concentram a maior proporção de churn (EDA seção 5) |
| `charges_per_tenure` | Monthly Charges / (Tenure + 1) | Pressão de preço relativa ao tempo de relacionamento |
| `tenure_bin` | Faixas: 0–12m / 13–24m / 25–48m / 49+m | Comportamento de churn muda por estágio de vida do cliente |

O `tenure_bin` foi testado em duas codificações alternativas:

| Variante | Codificação | Features totais | Detalhe |
|----------|-------------|-----------------|---------|
| `le` | Label encoding ordinal 0–3 | 32 | Tratado como numérico pelo StandardScaler |
| `ohe` | One-hot: 4 colunas binárias | 35 | Sem ordinalidade implícita — semântica correta para modelos não-lineares |

### Design da bateria

A bateria cruzou **2 splits × 3 variantes de FE × 5 modelos = 30 runs de baseline**
(mais 18 runs de MLP — cobertos no notebook 04):

| Dimensão | Valores |
|----------|---------|
| Split | 70/15/15 vs 80/10/10 |
| Feature set | `orig` (27 feat) / `le` (32 feat) / `ohe` (35 feat) |
| Modelos | Dummy, LogReg full, LogReg −MultiLines, LogReg −Phone, LogReg −Phone −ML |

Todos os 30 runs estão logados no MLflow. As células abaixo carregam e analisam os resultados.

In [ ]:
import re

# Load all 30 baseline battery runs from MLflow
battery_raw = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    max_results=500,
)

# Filter to battery runs: naming convention {model}_{split}_{tenure}
battery_mask = battery_raw["tags.mlflow.runName"].str.match(
    r"^(dummy|logreg[_\w]*)_(7015|8010)_(orig|le|ohe)$"
)
battery_df = battery_raw[battery_mask][
    ["tags.mlflow.runName", "metrics.roc_auc_mean", "metrics.holdout_val_roc_auc"]
].rename(columns={
    "tags.mlflow.runName": "run",
    "metrics.roc_auc_mean": "cv_auc",
    "metrics.holdout_val_roc_auc": "holdout_auc",
}).copy()

# Keep best run per name (re-runs produce duplicates)
battery_df = (
    battery_df
    .sort_values("cv_auc", ascending=False)
    .drop_duplicates(subset="run", keep="first")
    .reset_index(drop=True)
)

# Parse run name components
battery_df["split"] = battery_df["run"].str.extract(r"_(7015|8010)_")[0]
battery_df["tenure"] = battery_df["run"].str.extract(r"_(orig|le|ohe)$")[0]
battery_df["model"] = battery_df["run"].str.replace(r"_(7015|8010)_(orig|le|ohe)$", "", regex=True)

print(f"Battery baseline runs loaded: {len(battery_df)}")
battery_df.sort_values("holdout_auc", ascending=False).head(12).to_string(index=False)

### 9.1 Feature Engineering — `orig` vs `le` vs `ohe`

Feature engineering ajuda consistentemente: `le` e `ohe` superam `orig`
em ~+0.003 CV AUC e ~+0.002 holdout AUC. A diferença é pequena mas robusta —
aparece em todos os 5 modelos e em ambos os splits.

A tabela abaixo agrega a performance **média dos LogRegs** por variante de FE e split
(exclui Dummy). As colunas `delta_*` mostram o ganho relativo ao `orig` do mesmo split.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Average LogReg metrics by (split, tenure_variant) — excludes Dummy
logreg_df = battery_df[
    battery_df["model"].str.startswith("logreg")
].dropna(subset=["split", "tenure"])

fe_summary = (
    logreg_df
    .groupby(["split", "tenure"])[["cv_auc", "holdout_auc"]]
    .mean()
    .round(4)
    .reset_index()
)
fe_summary["split_label"] = fe_summary["split"].map({"7015": "70/15/15", "8010": "80/10/10"})

# Compute delta vs 'orig' within each split
for split_val in ["7015", "8010"]:
    orig_cv = fe_summary.loc[
        (fe_summary["split"] == split_val) & (fe_summary["tenure"] == "orig"), "cv_auc"
    ].values
    orig_holdout = fe_summary.loc[
        (fe_summary["split"] == split_val) & (fe_summary["tenure"] == "orig"), "holdout_auc"
    ].values
    if len(orig_cv):
        mask = fe_summary["split"] == split_val
        fe_summary.loc[mask, "delta_cv"] = (fe_summary.loc[mask, "cv_auc"] - orig_cv[0]).round(4)
        fe_summary.loc[mask, "delta_holdout"] = (fe_summary.loc[mask, "holdout_auc"] - orig_holdout[0]).round(4)

display(
    fe_summary[["split_label", "tenure", "cv_auc", "holdout_auc", "delta_cv", "delta_holdout"]]
    .sort_values(["split_label", "tenure"])
    .reset_index(drop=True)
)

In [ ]:
# Bar chart: CV AUC by FE variant for each split
tenure_order = ["orig", "le", "ohe"]
tenure_labels = ["Original\n(27 feat)", "LE\n(32 feat)", "OHE\n(35 feat)"]
colors = {"orig": "#90CAF9", "le": "#1E88E5", "ohe": "#0D47A1"}
split_titles = {"7015": "Split 70/15/15", "8010": "Split 80/10/10"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, split_val in zip(axes, ["7015", "8010"]):
    sub = fe_summary[fe_summary["split"] == split_val].set_index("tenure")
    vals = [float(sub.loc[t, "cv_auc"]) if t in sub.index else 0.0 for t in tenure_order]
    x = np.arange(len(tenure_order))
    bars = ax.bar(x, vals, color=[colors[t] for t in tenure_order], width=0.5, edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(tenure_labels)
    ax.set_title(split_titles[split_val], fontsize=11)
    ax.set_ylabel("ROC-AUC (CV médio — LogReg)")
    ax.set_ylim(0.845, 0.870)
    ax.axhline(0.8588, color="gray", linestyle="--", linewidth=1, label="Ref. Fase 2 (0.8588)")
    for bar, val in zip(bars, vals):
        ax.annotate(
            f"{val:.4f}",
            xy=(bar.get_x() + bar.get_width() / 2, val),
            xytext=(0, 4), textcoords="offset points",
            ha="center", fontsize=9,
        )

axes[0].legend(fontsize=8)
fig.suptitle("Impacto do Feature Engineering na AUC-ROC (média dos 4 LogRegs)", fontsize=12)
plt.tight_layout()
plt.show()

### 9.2 Análise de Split — 70/15/15 vs 80/10/10

O split 80/10/10 fornece ~700 amostras a mais de treino (+14%). Para o LogReg, onde a
capacidade do modelo não é limitante, o impacto se manifesta principalmente no holdout AUC
(+~0.020 vs 70/15/15 com a mesma variante de FE).

Para modelos com maior capacidade (MLP, Random Forest), essa diferença de treino tem potencial
para contribuir mais. Adicionalmente, com 80/10/10 é possível reservar os 10% finais como
**conjunto de teste cego** — nunca visto durante treino ou seleção de hiperparâmetros — o que
fornece uma estimativa de generalização mais conservadora e confiável.

In [ ]:
# Holdout AUC by split — best FE variant per split (LogReg only)
best_per_split = (
    fe_summary[fe_summary["tenure"] != "dummy"]
    .sort_values("holdout_auc", ascending=False)
    .groupby("split")
    .first()
    .reset_index()
)

splits_plot = ["7015", "8010"]
labels_plot = ["70/15/15", "80/10/10"]
vals_plot = [
    float(best_per_split.loc[best_per_split["split"] == s, "holdout_auc"].values[0])
    for s in splits_plot
]
bar_colors = ["#90CAF9", "#0D47A1"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels_plot, vals_plot, color=bar_colors, width=0.35, edgecolor="white")
ax.set_ylabel("ROC-AUC (holdout — melhor variante de FE)")
ax.set_ylim(0.840, 0.882)
ax.set_title("Split 80/10/10 produz holdout AUC superior", fontsize=11)
for bar, val in zip(bars, vals_plot):
    ax.annotate(
        f"{val:.4f}",
        xy=(bar.get_x() + bar.get_width() / 2, val),
        xytext=(0, 4), textcoords="offset points",
        ha="center", fontsize=11, fontweight="bold",
    )
plt.tight_layout()
plt.show()

print(f"Diferença de holdout AUC (8010 - 7015): {vals_plot[1] - vals_plot[0]:+.4f}")

### 9.3 Conclusão e Configuração Canônica

| Dimensão | Análise | Decisão |
|----------|---------|---------|
| Feature engineering | `le` e `ohe` superam `orig` em ~+0.003 CV e ~+0.002 holdout — diferença consistente em todos os modelos | **Usar FE** |
| `le` vs `ohe` | Performance praticamente idêntica (diff < 0.001); `ohe` tem semântica correta para RF (sem ordinalidade implícita nos bins) | **`ohe`** (35 feat) |
| Split | 80/10/10 produz holdout AUC ~+0.020 superior e fornece conjunto de teste cego para avaliação desenviesada | **80/10/10** |

**Melhor baseline encontrado:** `logreg_nophone_noml_8010_le`
— AUC-ROC CV = 0.8583, holdout = **0.8725**

**Configuração canônica para Fase 3** (MLP e Random Forest):
- Split: **80/10/10** (`test_size=0.10`, `val_size=0.10`)
- Feature engineering: **`ohe`** (`tenure_variant="ohe"`, 35 features)

Estas decisões estão formalizadas em [ADR-009](../docs/architecture.md#adr-009) e
[ADR-010](../docs/architecture.md#adr-010).

In [ ]:
# Top-5 baselines by holdout AUC — canonical reference for Phase 3
top5 = battery_df[~battery_df["model"].str.startswith("dummy")].sort_values(
    "holdout_auc", ascending=False
).head(5)

print("Top-5 baselines por holdout AUC:")
print(top5[["run", "cv_auc", "holdout_auc"]].to_string(index=False))

best_run = top5.iloc[0]
print(f"\nReferência Fase 3: {best_run['run']}")
print(f"  CV AUC    = {best_run['cv_auc']:.4f}")
print(f"  Holdout   = {best_run['holdout_auc']:.4f}")